In [6]:
import numpy as np
import h5py
import matplotlib.pyplot as plt
import trackio
from torch.utils.data import Dataset, DataLoader, random_split
import torch
import torch.nn as nn

keys = 'u', 'v', 'w', 'p_total'

def extract(real, keys):
    with h5py.File("/data1/dataToYouri/couette_dns_ml_f32.h5", 'r') as f:
        data = []
        for k in keys:
            data.append(np.array(f[real][k]))
        return np.stack(data, axis=1)

realizations = ['R13', 'R14', 'R15', 'R18', 'R19', 'R21', 'R22', 'R23', 'R26', 'R28', 'R3', 'R4', 'R5']
samplings = {real: np.s_[:-1:5000 // 300] if real in ["R3", "R4", "R5"] else np.s_[:] for real in realizations} 

raw_d = {real: extract(real, keys)[samplings[real]] for real in realizations}
all_data = np.concatenate(list(raw_d.values()), axis=0)
T, C, X, Y, Z = all_data.shape

# Test and val realizations choosen as they have roughly avg mean
val_realizations = ["R18", "R5"]
test_realizations = ["R3", "R14"]
train_realizations = [real for real in realizations if real not in val_realizations + test_realizations]
val_npds = np.concatenate([raw_d[r] for r in val_realizations])
test_npds = np.concatenate([raw_d[r] for r in test_realizations])
train_npds = np.concatenate([raw_d[r] for r in train_realizations])

device = "cuda"

class DS(Dataset):
    def __init__(self, d):
        self.x = torch.from_numpy(d[:, 3, :, 0, :]).float().reshape(-1, X * 1 * Z).to(device)
        self.y = torch.from_numpy(d[:, :3, :, :, :]).float().reshape(-1, X * Y * Z * 3).to(device)
    def __len__(self):
        return self.x.shape[0]
    def __getitem__(self, i):
        return self.x[i], self.y[i]

normalization_axes = (0, 2, 4,)
all_data_mean = all_data.mean(normalization_axes, keepdims=True)
all_data_std = all_data.std(normalization_axes, keepdims=True) + 1e-10

def normalize(data):
    return (data - all_data_mean) / all_data_std

def unnormalize_velocity(data):
    return (data * all_data_std[:, :3]) + all_data_mean[:, :3]


val_ds = DS(normalize(val_npds))
test_ds = DS(normalize(test_npds))
train_ds = DS(normalize(train_npds))

def train(model, opti, loss, ds):
    model.train()
    total_loss = 0
    for x, y in ds:
        opti.zero_grad()
        pred = model(x)
        l = loss(y, pred)
        total_loss += l.item()
        l.backward()
        opti.step()
    return total_loss / len(ds)

def evaluate(model, loss, ds):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for x, y in ds:
            pred = model(x)
            l = loss(y, pred)
            total_loss += l.item()
    return total_loss / len(ds)

def predict(model, flat_ds):
    model.eval()
    with torch.no_grad():
        x, y = flat_ds
        pred = model(x)
    return pred


def explained_variance(preds, ys):
    unmzd_preds = unnormalize_velocity(preds.detach().cpu().reshape(-1, 3, X, Y, Z))
    unmzd_y = unnormalize_velocity(ys.detach().cpu().reshape(-1, 3, X, Y, Z))
    unormalized_loss = torch.nn.functional.mse_loss(unmzd_preds, ys)
    explained_var = 1.0 - torch.nn.functional.mse_loss(unmzd_preds, ys) / unmzd_y.var(unbiased=False)
    return explained_var


def train_loop(model, loss, epochs, optim, lr, batch_size, project_name, device):
    
    opti = optim(model.parameters(), lr=lr)
    trackio.init(
        project=project_name,
        config={"epochs": epochs, "learning_rate": lr, "batch_size": batch_size}
    )
    for epoch in range(epochs):
        train_error = train(model, opti, loss, train_dl)
        test_error = test(model, loss, test_dl)
    
        trackio.log({
            "epoch": epoch,
            "train_error": train_error,
            "test_error": test_error,
        })
    
    trackio.finish()

batch_size = 4
train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
test_dl = DataLoader(test_ds, batch_size=batch_size, shuffle=False)
val_dl = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

# define model

epochs = 20
loss = nn.MSELoss(reduction='mean')
lr = 1e-3
model = nn.Sequential(nn.Linear(X * 1 * Z, X * Y * Z * 3)).to(device)
optim = torch.optim.Adam
project_name = "mlp-4"


# train_loop(model, loss, epochs, optim, lr, batch_size, project_name, device)

In [11]:
depth_stage_1 = 4


class m4(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        x = 

model = nn.Sequential(
    nn.Conv2d(1, 8, 3),
    nn.SiLU(),
    nn.Conv2d(8, 16, 3),
    nn.SiLU(),
    nn.Conv2d(16, 16, 3, padding="same"),
    nn.SiLU(),
    nn.View(),
    # nn.Linear(16 * 16, 8 * 3 * depth_stage_1), # dimension lift
    nn.SiLU(),
    nn.Conv3d
)


In [ ]:
train_loop(model, loss, epochs, optim, lr, batch_size, project_name, device)

In [ ]:
val_preds = predict(model, val_dl.dataset.x)
val_exp_var = explained_variance(val_preds, val_dl.dataset.y)
print(val_exp_var)